In [1]:
import math
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import os
import time
from tqdm import tqdm, trange

import sys
sys.path.insert(0, "/home/palakons/shapevae")#files are relative to singularity home, coz the server is running singularity from there
from model.ptv3_based_model import PointVAE, VAEConfig


from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from pytorch3d.loss import chamfer_distance
import numpy as np

import wandb




# autoreload py
%load_ext autoreload
%autoreload 2

/home/palakons/.conda/envs/pro_pt3d/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


/home/palakons/.conda/envs/pro_pt3d/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/palakons/.conda/envs/pro_pt3d/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


In [2]:

from experiment_runner import ExperimentConfig, run_training
from shapenet_dataset import ShapeNetDataset
from visualize import visualize_reconstructions, plot_pointclouds,visualize_interpolations

from model.ptv3_based_model import PointVAE, VAEConfig,check_voxel_collisions
from model.losses import repulsion_exp_loss,loss_fn_cd_plus_real_repulsion,loss_fn_cd,loss_fn_cd_plus_repulsion
from model.base_model import PointCloudAE
from model.base_model import PointCloudAE

In [3]:
# Shared configs for all runs
z_dim = 1024
batch_size = 1024
num_epochs = 100
lr = 1e-3
val_split = 0.1
seed = 42
run_root="/shapevae_weights"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Notebook-safe default: multiprocessing DataLoader workers can trigger
# "can only test a child process" cleanup errors in Jupyter.
num_workers = 0


device: cuda


In [4]:
def loadmodel_from_wandb(run_id, device,num_points=1024):
    api = wandb.Api()
    run_allcat = api.run(f"alephnir-vistec/shapevae/{run_id}")

    print("  run_dir:", run_allcat.summary["run_dir"].replace("/ist-nas/ist-share/vision/pratchp", ""))
    best_checkpoint_path = os.path.join(run_allcat.summary["run_dir"].replace("/ist-nas/ist-share/vision/pratchp", ""), "checkpoints/best.pt")

    # print("    Found checkpoint:", best_checkpoint_path)
    checkpoint = torch.load(best_checkpoint_path, map_location=device)
    # print(f"keys in checkpoint: {checkpoint.keys()}") #['epoch', 'best_val', 'model_state_dict', 'optimizer_state_dict', 'config']
    model_state_dict = checkpoint["model_state_dict"]
    if "ptv3" in run_allcat.name:
        cfg = VAEConfig(hidden_dim=64, latent_dim=128, num_points=num_points, variational=("vae" in run_allcat.name)   , grid_size=.01)
        point_ae_model = PointVAE(cfg=cfg).to(device)
    elif "baseline" in run_allcat.name:
        point_ae_model = PointCloudAE(z_dim=z_dim, num_points=num_points)
    else:
        print("    Unrecognized model type in run name. Skipping visualization.")

    point_ae_model.load_state_dict(model_state_dict)
    point_ae_model.to(device)
    point_ae_model.eval()
    return point_ae_model,run_allcat

def load_dataset(data_dir, object_classes,verbose=False,batch_size=16, num_workers=0, seed=42,split_ratios=(0.8, 0.1, 0.1),device="cuda"):
    torch.manual_seed(seed)
    if len(object_classes) == 0:
        object_classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
        print(f"Found object classes: {object_classes}")
    datasets = [ShapeNetDataset(data_dir=data_dir, object_class=obj_cls) for obj_cls in object_classes]
    dataset = torch.utils.data.ConcatDataset(datasets)

    all_indices = np.arange(len(dataset))
    np.random.shuffle(all_indices)
    train_idx, val_idx, test_idx = np.split(all_indices, [int(split_ratios[0]*len(dataset)), int((split_ratios[0]+split_ratios[1])*len(dataset))])
    train_set = Subset(dataset, train_idx)
    val_set = Subset(dataset, val_idx)
    test_set = Subset(dataset, test_idx)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers,
        pin_memory=(device == "cuda"),
    )
    if verbose:
        # plot 3d scatter of first batch
        for batch in train_loader:
            pcs = batch['points']  # shape (B, N, 3)
            ids = batch['object_id']
            pc_list = list(zip(ids, pcs))
            plot_pointclouds(pc_list, n_cols=8)
            break
    val_loader = DataLoader(
        val_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
    )
    test_loader = DataLoader(
        test_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
    )
    return train_loader, val_loader, test_loader, train_idx, val_idx, test_idx


from model.base_model import PointCloudAE

from model.ptv3_based_model import PointVAE, VAEConfig,AECLIPProjectionHead,train_adaptor

# count categories
from collections import Counter

def count_categories(loader):

    all_classes = []
    tt = tqdm(loader, desc="Counting categories")
    for batch in tt:
        class_labels = batch['category']
        all_classes.extend(class_labels)
        tt.set_description(f"{Counter(all_classes)}")
    return Counter(all_classes)

def train_clip2z_adaptor(  config,
    point_ae_model,
    adaptor_model,
    optimizer,
    loss_fn,
    train_loader,
    val_loader,
    device,
):
    '''
    gt: batch.clip_latent (B, 512)
    pred: model(batch.points) -> (B, 512)
    '''

    best_val_loss = float('inf')
    point_ae_model.eval()  # freeze point AE during adaptor training

    # outer bar: do not leave previous bars in notebook
    ppbar = tqdm(range(config.num_epochs), desc="Adaptor Training", leave=False, position=0)
    for epoch in ppbar:
        adaptor_model.train()

        train_loss = 0.0
        # inner training bar at position 1 so it replaces/clears correctly
        for batch in train_loader:
            points = batch['points'].to(device)  # (B, N, 3)
            clip_latent = batch['clip_latent'].to(device)  # (B, 512)

            optimizer.zero_grad()
            z = point_ae_model.encoder(points)  # (B, 1024)
            z_pred = adaptor_model(clip_latent)  # (B, 1024)
            loss = loss_fn(z_pred, z)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * points.size(0)

        avg_train_loss = train_loss / len(train_loader.dataset)

        # Validation
        adaptor_model.eval()
        val_loss = 0.0
        # validation bar at same position as training to avoid stacking
        with torch.no_grad():
            for batch in val_loader:
                points = batch['points'].to(device)  # (B, N, 3)
                clip_latent = batch['clip_latent'].to(device)  # (B, 512)

                z = point_ae_model.encoder(points)  # (B, 1024)
                z_pred = adaptor_model(clip_latent)  # (B, 1024)
                loss = loss_fn(z_pred, z)
                val_loss += loss.item() * points.size(0)

        avg_val_loss = val_loss / len(val_loader.dataset)



        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_path = os.path.join(config.run_root, f"best_clip2z_adaptor_{config.name}.pt")
            torch.save(adaptor_model.state_dict(), best_path)
    

        ppbar.set_description(f"Train:{avg_train_loss:.4f}, Val:{avg_val_loss:.4f}")
    #save model at end of training as well, for checkpointing
    final_path = os.path.join(config.run_root, f"final_clip2z_adaptor_{config.name}.pt")
    torch.save(adaptor_model.state_dict(), final_path)
    print(f"Best adaptor model saved at: {best_path} with val loss: {best_val_loss:.4f}")
    print(f"Final adaptor model saved at: {final_path}")

    return config.run_root, {'best_val_loss': best_val_loss}

In [5]:
import clip
model_name = "ViT-B/32"
clip_model, preprocess = clip.load(model_name, device=device, jit=False)
#coome up with chaor attribute

def best_worst_pointclouds_by_keyword(
    point_ae_model,
    model_adaptor,
    clip_model,
    keywords,
    train_loader,
    device,
):
    clip_model.eval()
    clip_model.to(device)
    model_adaptor.eval()

    kw_tokens = clip.tokenize(keywords).to(device)                   # (K,)
    kw_clip_latents = clip_model.encode_text(kw_tokens).detach()    # (K,512)
    kw_norm = F.normalize(kw_clip_latents, dim=1).float()                    # (K,512)

    overall_min_sim = [float('inf')] * len(keywords)
    overall_max_sim = [float('-inf')] * len(keywords)
    pc_max = [None] * len(keywords)
    pc_min = [None] * len(keywords)

    for batch in train_loader:
        points = batch['points'].to(device)  # (B, N, 3)

        z = point_ae_model.encoder(points)  # (B, 1024)
        pred_clip_latent = model_adaptor(z)  # (B, 512)
        
        pred_norm = F.normalize(pred_clip_latent, dim=1).float()     # (B,512)
        sims = pred_norm @ kw_norm.t()                      # (B, K)

        # for keyword index k, column = sims[:, k]
        for k_idx, keyword in enumerate(keywords):
            col = sims[:, k_idx]                           # (B,)
            batch_min_sim, batch_min_idx = torch.min(col, dim=0)
            batch_max_sim, batch_max_idx = torch.max(col, dim=0)
            if batch_min_sim < overall_min_sim[k_idx]:
                overall_min_sim[k_idx] = batch_min_sim.item()
                pc_min[k_idx] = points[batch_min_idx].cpu().numpy()  # (N, 3)
            if batch_max_sim > overall_max_sim[k_idx]:
                overall_max_sim[k_idx] = batch_max_sim.item()
                pc_max[k_idx] = points[batch_max_idx].cpu().numpy()  # (N, 3)
    return zip(overall_min_sim, pc_min), zip(overall_max_sim, pc_max)

In [ ]:

def eval_clip_similarity(
    point_ae_model,
    model_adaptor_z2clip,
    clip_model,
    keywords,
    dataloaders, #("train", train_loader_ptv3_allcat), ("val", val_loader_ptv3_allcat)
    device,
):
    clip_model.eval()
    clip_model.to(device)
    model_adaptor_z2clip.eval()

    kw_tokens = clip.tokenize(keywords).to(device)                   # (K,)
    kw_clip_latents = clip_model.encode_text(kw_tokens).detach()    # (K,512)
    kw_norm = F.normalize(kw_clip_latents, dim=1).float()                    # (K,512)
    results = {}
    for name, dataloader in dataloaders: 
        print(f"Evaluating {name}: finding best/worst pointclouds for keywords: {keywords}")
        min_results, max_results = best_worst_pointclouds_by_keyword(
            point_ae_model=point_ae_model,
            model_adaptor=model_adaptor_z2clip,
            clip_model=clip_model,
            keywords=keywords,
            train_loader=dataloader,
            device=device,
        )

        sim_points_pairs: Sequence[Tuple[str, torch.Tensor | np.ndarray]] = list(zip(keywords*2, [a[1] for a in list(min_results)+list(max_results)]))
        results[name] = sim_points_pairs

        print(f"Plotting for {name} set:")
        plot_pointclouds(sim_points_pairs, n_cols=len(keywords),truncate_length=20)
    return results

In [7]:
train_loader_allcat, val_loader_allcat, test_loader_allcat, train_idx_allcat, val_idx_allcat, test_idx_allcat = load_dataset(data_dir='/shapevae_preprocessed/sampled_pointcloud_1024pt_clipmesh_allcat', object_classes=[], verbose=False,batch_size=batch_size, num_workers=num_workers, seed=42,split_ratios=(0.8, 0.1, 0.1),device="cuda")

num_points = 1024

Found object classes: ['02747177', '02808440', '02818832', '02871439', '02933112', '03001627', '03211117', '04256520', '04379243']


## Train PTv3

In [ ]:
cfg = VAEConfig(hidden_dim=64, latent_dim=128, num_points=1024, variational=False, grid_size=.02)
model_ptv3_allcat = PointVAE(cfg=cfg)

optimizer_ptv3_allcat = torch.optim.Adam(model_ptv3_allcat.parameters(), lr=lr)

cfg_ptv3_allcat = ExperimentConfig(
    name=f"baseline_ptv3_allcat_4interp_{num_epochs}epoch",
    num_epochs=num_epochs,
    seed=seed,
    use_amp=False,
    save_every=10,
    run_root=run_root,
    epoch_log_every=10,
)

In [7]:

run_dir_ptv3_allcat, summary_ptv3_allcat, run_id_ptv3_allcat = run_training(
    config=cfg_ptv3_allcat,
    model=model_ptv3_allcat,
    optimizer=optimizer_ptv3_allcat,
    loss_fn=loss_fn_cd_plus_real_repulsion,
    train_loader=train_loader_ptv3_allcat,
    val_loader=val_loader_ptv3_allcat,
    device=device,
)

wandb: Currently logged in as: palakon-k to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[baseline_ptv3_allcat_4interp_100epoch] interpolation anchors selected: idx=[204, 1559, 1285, 2088], ids=['e6c900568268acf735836c728d324152', '7ab615debab4fff9afc85023f866b252', '3c475d9f0433a7eaad2650d014e970a5', '1f24b9a75606239466e24bbfdb446f55']


train:baseline_ptv3_allcat_4interp_100epoch:   0%|          | 0/2100 [00:00<?, ?it/s]

W0508 13:57:10.716000 2539 site-packages/torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


shape of points: torch.Size([1024, 1024, 3])
shape of backbone input - coord: torch.Size([1048576, 3]), feat: torch.Size([1048576, 64]), batch: torch.Size([1048576]), offset: torch.Size([1024])
feat_out shape: torch.Size([1024, 64]), batch shape: torch.Size([864019]), offset shape: torch.Size([1024])


IndexError: The shape of the mask [864019] at index 0 does not match the shape of the indexed tensor [1024, 64] at index 0

In [ ]:
visualize_reconstructions(
    model=model_ptv3_allcat,
    loader=val_loader_ptv3_allcat,
    device=device,
    num_batches=10,
    n_cols=10,
)

In [ ]:

print("Loading run_id_ptv3_allcat:", run_id_ptv3_allcat)
point_ae_model_ptv3_allcat,run_ptv3 =  loadmodel_from_wandb(run_id_ptv3_allcat.split('/')[-1], device)

In [ ]:
# call visualization on the 4 anchors, interpolating between them in pairs (0-1, 1-2, 2-3, 3-0)
visualize_interpolations(
    model=point_ae_model_ptv3_allcat,
    loader=diverse_val_loader_ptv3_allcat,
    device=device,
    grid_size=(5, 5),  # 4 pairs of interpolations
    input_color="dodgerblue",
    interp_color="orangered",
    input_alpha=0.85,
    interp_alpha=0.85,
    title="Interpolations between 4 Diverse Anchor Chairs",
)

### Visualize each AE model

In [8]:
wandb_allcat_run_ids = ["yy387w7f","9y2o6vi2","4a9nwl03","pmurdgio"]

In [ ]:
for main_model_id in wandb_allcat_run_ids:
    point_ae_model,wandb_run = loadmodel_from_wandb(main_model_id, device,num_points=num_points)

    visualize_interpolations(point_ae_model, val_loader_allcat, device=device, grid_size=(5, 5), input_color="dodgerblue", interp_color="orangered", input_alpha=0.85, interp_alpha=0.85, title=f"Interpolations for model {main_model_id}")

### Train Clip2Z & z2Clip adaptors

In [ ]:
for main_model_id in wandb_allcat_run_ids:
    point_ae_model,wandb_run = loadmodel_from_wandb(main_model_id, device,num_points=num_points)

    model_adaptor_z2clip = AECLIPProjectionHead(input_dim=z_dim, output_dim=512).to(device)
    optimizer_adaptor_z2clip = torch.optim.Adam(model_adaptor_z2clip.parameters(), lr=lr)

    cfg_adaptor_z2clip = ExperimentConfig(
        name=f"adaptor_z2clip_{main_model_id}",
        num_epochs=num_epochs,
        seed=seed,
        use_amp=False,
        save_every=10,
        run_root=run_root,)

    print(f"Training z2clip adaptor for run: {wandb_run.name}")
    run_dir_adaptor_z2clip, summary_adaptor_z2clip = train_adaptor(
        config=cfg_adaptor_z2clip,
        point_ae_model=point_ae_model,
        adaptor_model=model_adaptor_z2clip,
        optimizer=optimizer_adaptor_z2clip,
        loss_fn=nn.MSELoss(),
        train_loader=train_loader_allcat,
        val_loader=val_loader_allcat,
        device=device,
    )

wandb: Currently logged in as: palakon-k to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  run_dir: /shapevae_weights/20260507-102556_palakons_baseline_cd_allcat_4interp_1000epoch
Training z2clip adaptor for run: baseline_cd_allcat_4interp_1000epoch


Best adaptor model saved at: /shapevae_weights/best_adaptor_adaptor_z2clip_yy387w7f.pt with val loss: 0.0211
Final adaptor model saved at: /shapevae_weights/final_adaptor_adaptor_z2clip_yy387w7f.pt
  run_dir: /shapevae_weights/20260507-172935_palakons_baseline_cd_allcat_4interp_100epoch
Training z2clip adaptor for run: baseline_cd_allcat_4interp_100epoch


Train:0.0224, Val:0.0227:  39%|█████████████████████████████████████████████████████████████████████████▋                                                                                                                   | 39/100 [29:54<2:25:01, 142.64s/it]

In [18]:
%ls /shapevae_weights/final_clip2z*

/shapevae_weights/final_clip2z_adaptor.pt
/shapevae_weights/final_clip2z_adaptor_baseline_adaptor_clip2z_allcat.pt


In [ ]:
#train and val
keywords = ["a bean-bag-like chair","an oval chair", "a pointy chair","a spiky chair","a non-spiky chair", "a  modern chair", "a  thin chair", "a  thick chair"]

for main_model_id in wandb_allcat_run_ids:
    point_ae_model,wandb_run = loadmodel_from_wandb(main_model_id, device,num_points=num_points)
    #load z2clip adaptor
    model_adaptor_z2clip = AECLIPProjectionHead(input_dim=z_dim, output_dim=512).to(device)
    adaptor_path = os.path.join(run_root, f"final_adaptor_adaptor_z2clip_{main_model_id}.pt")
    print(f"Loading adaptor from: {adaptor_path}")

    model_adaptor_z2clip.load_state_dict(torch.load(adaptor_path, map_location=device))


    clip_eval_results = eval_clip_similarity(
        point_ae_model=point_ae_model,
        model_adaptor_z2clip=model_adaptor_z2clip,
        clip_model=clip_model,
        keywords=keywords,
        dataloaders=[("train", train_loader_allcat), ("val", val_loader_allcat)],
        device=device,
    )

  run_dir: /shapevae_weights/20260507-102556_palakons_baseline_cd_allcat_4interp_1000epoch
Loading adaptor from: /shapevae_weights/final_adaptor_adaptor_z2clip_yy387w7f.pt


KeyboardInterrupt: 

### Train clip2Z forinterpolation


In [ ]:
for main_model_id in wandb_allcat_run_ids:
    point_ae_model,wandb_run = loadmodel_from_wandb(main_model_id, device,num_points=num_points)

    model_adaptor_clip2z = AECLIPProjectionHead(input_dim=512, output_dim=z_dim).to(device)
    optimizer_adaptor_clip2z = torch.optim.Adam(model_adaptor_clip2z.parameters(), lr=lr)

    cfg_adaptor_clip2z = ExperimentConfig(
    name=f"adaptor_clip2z_{main_model_id}",
    num_epochs=num_epochs ,
    seed=seed,
    use_amp=False,
    save_every=10,
    run_root=run_root,)



    print(f"Training clip2z adaptor for run: {wandb_run.name}")
    run_dir_adaptor_clip2z, summary_adaptor_clip2z = train_clip2z_adaptor(
        config=cfg_adaptor_clip2z,
        point_ae_model=point_ae_model,  # use the same model as point AE since we're only training a small projection head on top
        adaptor_model=model_adaptor_clip2z,
        optimizer=optimizer_adaptor_clip2z,
        loss_fn=nn.MSELoss(),
        train_loader=train_loader_allcat,
        val_loader=val_loader_allcat,
        device=device,
    )

  run_dir: /shapevae_weights/20260507-102556_palakons_baseline_cd_allcat_4interp_1000epoch
Training clip2z adaptor for run: baseline_cd_allcat_4interp_1000epoch


Train:0.0002, Val:0.0002:   2%|███▊                                                                                                                                                                                          | 2/100 [05:19<3:51:53, 141.97s/it]